# ViSpoofDB 10K — Pipeline Thu Thập Dữ Liệu
## Nhóm 17 — Viet-Guard

## Cell 1 — Cài đặt

In [ ]:
import subprocess, sys

def pip_quiet(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_quiet("groq", "edge-tts", "nest_asyncio", "gtts", "pydub")

print("✅ Cài đặt xong: groq | edge-tts | nest-asyncio | gtts | pydub")


## Cell 2 — Điền API Key

| Nguồn | Free tier | Ghi chú |
|---|---|---|
| **FPT.AI** | ~2,000 ký tự/ngày | 8 tài khoản → 16,000 ký tự/ngày |
| **Viettel AI** | free tier | 1 tài khoản/người |
| **ElevenLabs** | 10,000 ký tự/tháng | cần email khác để tạo thêm tài khoản |
| **MiniMax** | ~10M token free khi signup | sau đó tính phí — đủ cho project này |
| **Zalo AI** | 1,000 req/ngày | đăng ký tại zaloai.vn → miễn phí |


In [ ]:
# ════════════════════════════════════════════════════════════════
#  ĐIỀN API KEY VÀO ĐÂY
#  - Điền nhiều key (list) để tự động xoay vòng khi hết quota
#  - Để "YOUR_..." nếu chưa có → tự bỏ qua nguồn đó
# ════════════════════════════════════════════════════════════════

# ── Groq (sinh corpus, miễn phí) ────────────────────────────────
GROQ_API_KEY = ""

# ── FPT.AI (~2,000 ký tự/ngày/tài khoản) ───────────────────────
FPTAI_KEYS = [
]

# ── Viettel AI — (api_key, token) ───────────────────────────────
VIETTEL_KEYS = [
]

# ── ElevenLabs (~10,000 ký tự/tháng/tài khoản) ─────────────────
ELEVENLABS_KEYS = [
]

# ── MiniMax (~10M token free khi signup) ────────────────────────
MINIMAX_KEYS = [
]

# ── Zalo AI (1,000 req/ngày free) ───────────────────────────────
ZALOAI_KEYS = [
]

print("✅ API keys loaded")
print(f"   FPT.AI     : {len(FPTAI_KEYS)} keys → ~{len(FPTAI_KEYS)*2000} ký tự/ngày")
print(f"   Viettel    : {len(VIETTEL_KEYS)} keys")
print(f"   ElevenLabs : {len(ELEVENLABS_KEYS)} keys → ~{len(ELEVENLABS_KEYS)*10000} ký tự/tháng")
print(f"   MiniMax    : {len(MINIMAX_KEYS)} keys (dùng free credits)")
print(f"   Zalo AI    : {len([k for k in ZALOAI_KEYS if k != 'YOUR_ZALOAI_KEY'])} keys sẵn sàng")


## Cell 3 — Import & Cấu hình

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, json, time, asyncio, io, requests, scipy
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import nest_asyncio

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from scipy.io.wavfile import write as write_wav
from groq import Groq

nest_asyncio.apply()

# ── Thư mục
BASE_DIR      = "/kaggle/working"
DATA_DIR      = f"{BASE_DIR}/data"
CORPUS_FILE   = f"{BASE_DIR}/corpus.txt"
METADATA_FILE = f"{BASE_DIR}/metadata.csv"
REF_VOICE_DIR = f"{BASE_DIR}/ref_voices"

# ── Thông số audio chuẩn
TARGET_SR       = 16000
TARGET_DURATION = 5.0
TARGET_LEN      = int(TARGET_SR * TARGET_DURATION)

# ── Số mẫu mỗi nguồn
N_CORPUS     = 1200
N_MMSTTS     = 1200
N_EDGETTS    = 1200
N_FPTAI      = 1200
N_VIETTEL    = 1200
N_ELEVENLABS = 1200
N_MINIMAX    = 1200
N_ZALOAI     = 1200
N_GTTS       = 2000

# ── Nguồn UNSEEN: không đưa vào training, chỉ dùng test
UNSEEN_SOURCES = {"gtts"}

# ── Mapping nguồn → kỹ thuật
TECHNIQUE_MAP = {
    "vivos":      "natural",
    "vlsp":       "natural",
    "mmstts":     "neural_tts",
    "edgetts":    "neural_tts_commercial",
    "fptai":      "neural_tts_commercial",
    "viettel":    "neural_tts_commercial",
    "elevenlabs": "voice_clone",
    "minimax":    "neural_tts_multilingual",
    "zaloai":     "neural_tts_commercial",
    "gtts":       "concatenative_tts",
}

print("✅ Config OK")
print(f"   torch : {torch.__version__}")
print(f"   numpy : {np.__version__}")
print(f"   GPU   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU   : {torch.cuda.get_device_name(0)}")


## Cell 4 — Danh sách giọng đa dạng

In [ ]:
EDGETTS_VOICES = [
    "vi-VN-HoaiMyNeural",
    "vi-VN-NamMinhNeural",
]

FPTAI_VOICES = [
    "banmai", "thuminh", "leminh", "minhquang",
    "lannhi", "ngochuyen", "giahuy", "anhdung",
]

VIETTEL_VOICES = [
    "hn-female-thuhuong-vdts-48k-mb-vc",
    "hn-male-manhdung-vdts-48k-mb-vc",
    "hn-female-thungan-vdts-48k-mb-vc",
    "hn-male-thetuan-vdts-48k-mb-vc",
    "sg-female-thaotrinh-vdts-48k-mb-vc",
    "sg-male-minhhoang-vdts-48k-mb-vc",
    "sg-female-ngochuyen-vdts-48k-mb-vc",
    "sg-male-namkhanh-vdts-48k-mb-vc",
]

MINIMAX_VOICES = [
    "Serene_Woman", "Wise_Woman", "Friendly_Person", "Deep_Voice_Man",
    "Lively_Girl",  "Calm_Woman", "Energetic_Man",   "Gentle_Man",
]

ZALOAI_SPEAKER_IDS = [1, 2, 3, 4, 5]

print("✅ Danh sách giọng:")
print(f"   edge-tts  : {len(EDGETTS_VOICES)} giọng VI")
print(f"   FPT.AI    : {len(FPTAI_VOICES)} giọng VI")
print(f"   Viettel   : {len(VIETTEL_VOICES)} giọng VI")
print(f"   MiniMax   : {len(MINIMAX_VOICES)} giọng (multilingual)")
print(f"   Zalo AI   : {len(ZALOAI_SPEAKER_IDS)} speaker ID VI")


## Cell 5 — Helper functions & Setup thư mục

In [ ]:
class KeyRotator:
    """Quản lý nhiều API key, tự động chuyển khi hết quota."""
    def __init__(self, keys, name="API"):
        self.keys = list(keys)
        self.name = name
        self.idx  = 0
        self.dead = set()

    @property
    def current(self):
        return self.keys[self.idx]

    @property
    def alive_count(self):
        return len(self.keys) - len(self.dead)

    def rotate(self, reason=""):
        self.dead.add(self.idx)
        print(f"  🔄 [{self.name}] Key {self.idx} hết ({reason})")
        for i in range(len(self.keys)):
            nxt = (self.idx + 1 + i) % len(self.keys)
            if nxt not in self.dead:
                self.idx = nxt
                print(f"     → Chuyển sang key {self.idx}")
                return True
        print(f"  ❌ [{self.name}] Tất cả {len(self.keys)} key đã hết quota!")
        return False

    def is_exhausted(self):
        return len(self.dead) >= len(self.keys)

    def should_rotate(self, status_code, resp_text=""):
        txt = (resp_text or "").lower()
        if status_code in (401, 403):                    return True, "key hết hạn/sai"
        if status_code == 429:                           return True, "rate limit / hết quota"
        if "quota" in txt or "limit exceeded" in txt:   return True, "hết quota"
        if "expired" in txt or "invalid key" in txt:    return True, "key không hợp lệ"
        return False, ""


def setup_dirs():
    for d in [
        REF_VOICE_DIR,
        f"{DATA_DIR}/real/vivos",
        f"{DATA_DIR}/real/vlsp",
        f"{DATA_DIR}/fake/mmstts",
        f"{DATA_DIR}/fake/edgetts",
        f"{DATA_DIR}/fake/fptai",
        f"{DATA_DIR}/fake/viettel",
        f"{DATA_DIR}/fake/elevenlabs",
        f"{DATA_DIR}/fake/minimax",
        f"{DATA_DIR}/fake/zaloai",
        f"{DATA_DIR}/fake/gtts",
        f"{DATA_DIR}/processed",
    ]:
        os.makedirs(d, exist_ok=True)
    print("✅ Thư mục sẵn sàng")


def n_existing(out_dir, ext="*.wav"):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    return len(list(Path(out_dir).glob(ext)))


def preprocess_file(args):
    in_path, out_path = args
    try:
        audio, _ = librosa.load(in_path, sr=TARGET_SR, mono=True)
        audio, _ = librosa.effects.trim(audio, top_db=20)
        if len(audio) < TARGET_LEN:
            audio = librosa.util.fix_length(audio, size=TARGET_LEN)
        else:
            audio = audio[:TARGET_LEN]
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        sf.write(out_path, audio, TARGET_SR)
        return True, in_path
    except Exception as e:
        return False, f"{in_path}: {e}"


setup_dirs()


## Cell 6 — Sinh corpus tiếng Việt (Groq, miễn phí)

In [ ]:
TOPICS = [
    "tin tức thời sự hàng ngày",
    "hội thoại mua bán, chợ búa",
    "hướng dẫn nấu ăn món Việt",
    "mô tả phong cảnh thiên nhiên Việt Nam",
    "câu chuyện gia đình, tình cảm",
    "thông báo, quảng cáo sản phẩm",
    "hỏi đường, chỉ đường trong thành phố",
    "đặt hàng online, giao hàng",
    "sức khỏe, khám bệnh, thuốc men",
    "học tập, trường lớp, thi cử",
    "thể thao, bóng đá, giải trí",
    "du lịch, khách sạn, đặt phòng",
    "thời tiết, dự báo khí tượng",
    "ngân hàng, chuyển khoản, tài chính",
    "cảnh báo lừa đảo, an ninh mạng",
    "cảm xúc vui mừng, hạnh phúc",
    "cảm xúc buồn bã, lo lắng, sợ hãi",
    "kể chuyện cổ tích, truyện ngắn Việt Nam",
    "ca dao, tục ngữ, thành ngữ Việt Nam",
    "kỹ thuật, công nghệ, điện thoại thông minh",
    "y tế, bệnh viện, cấp cứu",
    "nông nghiệp, làng quê Việt Nam",
    "ẩm thực, món ăn ba miền",
    "lịch sử, văn hóa, phong tục Việt Nam",
]

def generate_corpus(n_sentences=N_CORPUS):
    if os.path.exists(CORPUS_FILE):
        with open(CORPUS_FILE, encoding="utf-8") as f:
            sents = [l.strip() for l in f if l.strip()]
        print(f"✅ Corpus đã có: {len(sents)} câu")
        return sents

    if GROQ_API_KEY.startswith("YOUR_"):
        print("⚠️  Chưa điền GROQ_API_KEY")
        return []

    client    = Groq(api_key=GROQ_API_KEY)
    all_sents = []
    batch     = 0

    print(f"🔄 Sinh {n_sentences} câu...")

    while len(all_sents) < n_sentences and batch < n_sentences:
        topic  = TOPICS[batch % len(TOPICS)]
        needed = min(50, n_sentences - len(all_sents))
        try:
            resp = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content":
                    f'Tạo đúng {needed} câu tiếng Việt về chủ đề: "{topic}". '
                    f'Yêu cầu: 10-20 từ/câu, đủ 6 thanh điệu, tự nhiên, không trùng. '
                    f'Chỉ trả JSON, không markdown: {{"sentences":["câu 1","câu 2",...]}}'
                }],
                response_format={"type": "json_object"}
            )
            data   = json.loads(resp.choices[0].message.content)
            sents  = [s.strip() for s in data.get("sentences", [])
                      if s.strip() and s.strip() not in set(all_sents)]
            all_sents.extend(sents[:needed])
            batch += 1
            print(f"  Batch {batch} [{topic[:25]}]: +{len(sents)} | tổng={len(all_sents)}/{n_sentences}")
        except Exception as e:
            print(f"  ⚠️  Batch {batch+1} lỗi: {e} — thử lại sau 5s")
            time.sleep(5)
            batch += 1
        time.sleep(0.5)

    all_sents = all_sents[:n_sentences]
    with open(CORPUS_FILE, "w", encoding="utf-8") as f:
        f.write("\n".join(all_sents))
    print(f"✅ Lưu {len(all_sents)} câu → {CORPUS_FILE}")
    return all_sents

texts = generate_corpus()
print(f"Sẵn sàng: {len(texts)} câu")


## Cell 7 — Nguồn 1: MMS-TTS (Facebook)
> `facebook/mms-tts-vie` — tiếng Việt, hoàn toàn miễn phí, chạy local

In [ ]:
def collect_mmstts(texts, out_dir=f"{DATA_DIR}/fake/mmstts", n=N_MMSTTS):
    from transformers import VitsModel, AutoTokenizer
    import scipy.io.wavfile

    existing = n_existing(out_dir)
    if existing >= n:
        print(f"✅ MMS-TTS: đã có {existing}/{n} file")
        return

    print("🔄 MMS-TTS: load model facebook/mms-tts-vie (lần đầu ~1 phút)...")
    tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-vie")
    model     = VitsModel.from_pretrained("facebook/mms-tts-vie")
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model  = model.to(device)
    sr     = model.config.sampling_rate
    print(f"✅ MMS-TTS ready | sr={sr} | device={device}")

    for i, text in enumerate(texts[:n]):
        out_path = f"{out_dir}/mmstts_{i:04d}.wav"
        if os.path.exists(out_path):
            continue
        try:
            inputs = tokenizer(str(text).strip()[:200], return_tensors="pt").to(device)
            with torch.no_grad():
                output = model(**inputs).waveform
            wav = output.squeeze().cpu().numpy()
            scipy.io.wavfile.write(out_path, sr, wav)
        except Exception as e:
            print(f"  ⚠️  MMS-TTS [{i}]: {e}")
        if i % 100 == 0:
            print(f"  MMS-TTS: {i+1}/{n}")
            if device == "cuda": torch.cuda.empty_cache()

    print(f"✅ MMS-TTS xong: {n_existing(out_dir)} file → {out_dir}")

collect_mmstts(texts)


## Cell 8 — Nguồn 2: edge-tts (Microsoft, hoàn toàn miễn phí)
> Không cần tài khoản | 2 giọng VI xen kẽ

In [ ]:
async def _edgetts_async(texts, out_dir, n):
    import edge_tts
    for i, text in enumerate(texts[:n]):
        out_path = f"{out_dir}/edgetts_{i:04d}.mp3"
        if os.path.exists(out_path):
            continue
        voice = EDGETTS_VOICES[i % len(EDGETTS_VOICES)]
        for attempt in range(3):
            try:
                await edge_tts.Communicate(str(text).strip()[:300], voice).save(out_path)
                break
            except Exception as e:
                if attempt == 2: print(f"  ⚠️  edge-tts [{i}]: {e}")
                await asyncio.sleep(2 + attempt)
        if i % 100 == 0:
            print(f"  edge-tts: {i+1}/{n} | voice={voice}")
        await asyncio.sleep(0.3)

def collect_edgetts(texts, out_dir=f"{DATA_DIR}/fake/edgetts", n=N_EDGETTS):
    existing = n_existing(out_dir, "*.mp3")
    if existing >= n:
        print(f"✅ edge-tts: đã có {existing}/{n} file")
        return
    print(f"🔄 edge-tts: {n} mẫu | {len(EDGETTS_VOICES)} giọng")
    # nest_asyncio đã apply → asyncio.run() an toàn trên Kaggle Python 3.10+
    try:
        loop = asyncio.get_running_loop()
        loop.run_until_complete(_edgetts_async(texts, out_dir, n))
    except RuntimeError:
        asyncio.run(_edgetts_async(texts, out_dir, n))
    print(f"✅ edge-tts xong: {n_existing(out_dir, '*.mp3')} file")

collect_edgetts(texts)


## Cell 9 — Nguồn 3: FPT.AI (~2,000 ký tự/ngày × 8 tài khoản)
> Chạy mỗi ngày 1 lần, tăng `START_IDX_FPTAI` +40 mỗi ngày

In [ ]:
# ── THAY ĐỔI MỖI NGÀY ──────────────────────
# Ngày 1: 0 | Ngày 2: 40 | Ngày 3: 80 | ...
START_IDX_FPTAI = 0
# ─────────────────────────────────────────────

def collect_fptai(texts, out_dir=f"{DATA_DIR}/fake/fptai",
                  n=N_FPTAI, start_idx=0):
    valid_keys = [k for k in FPTAI_KEYS if k and not k.startswith("YOUR_")]
    if not valid_keys:
        print("⏭️  FPT.AI: chưa có key → fpt.ai/tts")
        return

    rotator = KeyRotator(valid_keys, "FPT.AI")
    batch   = texts[start_idx:start_idx + 40]
    if not batch:
        print("✅ FPT.AI: đã xong toàn bộ")
        return

    done_before = n_existing(out_dir, "*.mp3")
    session     = requests.Session()
    print(f"🔄 FPT.AI: câu {start_idx}→{start_idx+len(batch)} | "
          f"{len(valid_keys)} keys | {len(FPTAI_VOICES)} giọng")

    for i, text in enumerate(batch):
        if rotator.is_exhausted():
            print("❌ FPT.AI: tất cả key đã hết quota — dừng")
            break

        real_idx = start_idx + i
        out_path = f"{out_dir}/fptai_{real_idx:04d}.mp3"
        if os.path.exists(out_path) and os.path.getsize(out_path) > 5000:
            continue

        text_str = str(text).strip()
        if not text_str or text_str.isdigit() or len(text_str) < 5:
            print(f"  ⚠️  [{real_idx}] text không hợp lệ: '{text_str[:30]}' — bỏ qua")
            continue

        voice   = FPTAI_VOICES[real_idx % len(FPTAI_VOICES)]
        success = False

        for attempt in range(len(valid_keys) + 1):
            if rotator.is_exhausted(): break
            try:
                r = session.post(
                    "https://api.fpt.ai/hmi/tts/v5",
                    headers={"api-key":      rotator.current,
                             "voice":        voice,
                             "speed":        "",
                             "Content-Type": "application/json"},
                    json={"text": text_str[:300]},
                    timeout=15
                )
                nr, reason = rotator.should_rotate(r.status_code, r.text)
                if nr:
                    if not rotator.rotate(reason): break
                    continue

                try:
                    rj = r.json()
                except Exception:
                    rj = {}
                audio_url = rj.get("async", "")
                if not audio_url:
                    if not rotator.rotate("async URL rỗng — hết quota?"): break
                    continue

                for wait in range(20):
                    time.sleep(4)
                    try:
                        resp = session.get(audio_url, timeout=20)
                        ct   = resp.headers.get("Content-Type", "")
                        if (resp.status_code == 200
                                and "audio" in ct
                                and len(resp.content) > 5000):
                            with open(out_path, "wb") as f:
                                f.write(resp.content)
                            print(f"  [{real_idx}] OK | voice={voice} key={rotator.idx} ({(wait+1)*4}s)")
                            success = True
                            break
                    except Exception:
                        pass

                if success: break
                if not success:
                    print(f"  ⚠️  [{real_idx}] Timeout 80s — thử key khác")
                    if not rotator.rotate("timeout"): break

            except Exception as e:
                print(f"  ⚠️  [{real_idx}] attempt {attempt}: {e}")
                time.sleep(3)

        if not success:
            print(f"  ⚠️  [{real_idx}] Bỏ qua")
        time.sleep(1)

    done_after = n_existing(out_dir, "*.mp3")
    print(f"✅ FPT.AI hôm nay: +{done_after-done_before} | tổng={done_after}/{n}")
    print(f"   Ngày mai: START_IDX_FPTAI = {start_idx + 40}")
    print(f"   Key còn sống: {rotator.alive_count}/{len(valid_keys)}")
    missed = sorted(
        set(range(start_idx, start_idx + len(batch))) -
        {int(f.stem.split("_")[1]) for f in Path(out_dir).glob("fptai_*.mp3")}
    )
    if missed: print(f"   ⚠️  Cần retry: {missed}")

collect_fptai(texts, start_idx=START_IDX_FPTAI)


## Cell 10 — Nguồn 4: Viettel AI (free tier)
> Chạy mỗi ngày 1 lần, tăng `START_IDX_VIETTEL` +50 mỗi ngày

In [ ]:
START_IDX_VIETTEL = 0

_VIETTEL_ENDPOINTS = [
    "https://viettelai.vn/tts/speech_synthesis",
    "https://api.viettelai.vn/tts/speech_synthesis",
    "https://viettelgroup.ai/voice/api/tts/v1/rest/syn",
]

def _find_viettel_endpoint(api_key, token):
    for ep in _VIETTEL_ENDPOINTS:
        try:
            r = requests.post(
                ep,
                headers={"token": token, "api_key": api_key,
                         "Content-Type": "application/json"},
                json={"text": "xin chào",
                      "voice": "hn-female-thuhuong-vdts-48k-mb-vc",
                      "speed": 1.0, "tts_return_option": 2},
                timeout=10
            )
            if r.status_code == 200 and len(r.content) > 500:
                print(f"  ✅ Endpoint OK: {ep}")
                return ep
            print(f"  ✗ {ep} → {r.status_code} {r.text[:60]}")
        except Exception as e:
            print(f"  ✗ {ep} → {e}")
    return None

def collect_viettel(texts, out_dir=f"{DATA_DIR}/fake/viettel",
                    n=N_VIETTEL, start_idx=0):
    valid_keys = [(k, t) for k, t in VIETTEL_KEYS
                  if k and not k.startswith("YOUR_")]
    if not valid_keys:
        print("⏭️  Viettel: chưa có key → viettelai.vn")
        return

    rotator = KeyRotator(valid_keys, "Viettel")
    print("🔍 Tìm Viettel endpoint...")
    api_key, token = rotator.current
    endpoint = _find_viettel_endpoint(api_key, token)
    if not endpoint:
        print("❌ Không tìm được endpoint Viettel — kiểm tra key/token")
        r = requests.post(
            _VIETTEL_ENDPOINTS[0],
            headers={"token": token, "api_key": api_key,
                     "Content-Type": "application/json"},
            json={"text": "xin chào", "voice": VIETTEL_VOICES[0],
                  "speed": 1.0, "tts_return_option": 2}
        )
        print(f"   Status: {r.status_code}")
        print(f"   Response: {r.text[:300]}")
        return

    batch = texts[start_idx:start_idx + 50]
    if not batch:
        print("✅ Viettel: đã xong toàn bộ")
        return

    done_before = n_existing(out_dir)
    print(f"🔄 Viettel: câu {start_idx}→{start_idx+len(batch)} | "
          f"{len(valid_keys)} keys | {len(VIETTEL_VOICES)} giọng")

    for i, text in enumerate(batch):
        if rotator.is_exhausted():
            print("❌ Viettel: tất cả key hết quota — dừng")
            break

        real_idx = start_idx + i
        out_path = f"{out_dir}/viettel_{real_idx:04d}.wav"
        if os.path.exists(out_path) and os.path.getsize(out_path) > 500:
            continue

        voice   = VIETTEL_VOICES[real_idx % len(VIETTEL_VOICES)]
        success = False

        for attempt in range(len(valid_keys) + 1):
            if rotator.is_exhausted(): break
            api_key, token = rotator.current
            try:
                r = requests.post(
                    endpoint,
                    headers={"token": token, "api_key": api_key,
                             "Content-Type": "application/json"},
                    json={"text": str(text).strip()[:300],
                          "voice": voice, "speed": 1.0, "tts_return_option": 2},
                    timeout=15
                )
                nr, reason = rotator.should_rotate(r.status_code, r.text)
                if nr:
                    if not rotator.rotate(reason): break
                    continue
                if r.status_code == 200 and len(r.content) > 500:
                    with open(out_path, "wb") as f: f.write(r.content)
                    region = "HN" if voice.startswith("hn") else "SG"
                    gender = "F" if "female" in voice else "M"
                    print(f"  [{real_idx}] OK | {region}-{gender} key={rotator.idx}")
                    success = True; break
                else:
                    print(f"  ⚠️  [{real_idx}] {r.status_code} {r.text[:80]}")
                    if r.status_code == 404: break
            except Exception as e:
                print(f"  ⚠️  [{real_idx}]: {e}")
                time.sleep(3)

        time.sleep(1.5)

    done_after = n_existing(out_dir)
    print(f"✅ Viettel hôm nay: +{done_after-done_before} | tổng={done_after}/{n}")
    print(f"   Ngày mai: START_IDX_VIETTEL = {start_idx + 50}")
    print(f"   Key còn sống: {rotator.alive_count}/{len(valid_keys)}")

collect_viettel(texts, start_idx=START_IDX_VIETTEL)


## Cell 11 — Nguồn 5: ElevenLabs (~10,000 ký tự/tháng)
> Tự động crawl giọng tiếng Việt | key rotation tự động

In [ ]:
def _get_vi_voices_elevenlabs(api_key):
    vi_voices = []
    try:
        r = requests.get("https://api.elevenlabs.io/v1/voices",
                         headers={"xi-api-key": api_key}, timeout=10)
        if r.status_code == 200:
            for v in r.json().get("voices", []):
                labels = v.get("labels") or {}
                lang   = labels.get("language", "").lower()
                desc   = (v.get("description") or "").lower()
                name   = v.get("name", "").lower()
                if any(kw in lang + desc + name
                       for kw in ["vietnamese", "việt", "viet", "vi-vn"]):
                    vi_voices.append({"voice_id": v["voice_id"], "name": v["name"]})
    except Exception as e:
        print(f"  ⚠️  Lỗi my voices: {e}")

    page = 0
    while len(vi_voices) < 10 and page < 5:
        try:
            r = requests.get(
                "https://api.elevenlabs.io/v1/shared-voices",
                headers={"xi-api-key": api_key},
                params={"language": "vi", "page_size": 100, "page": page},
                timeout=10
            )
            if r.status_code != 200: break
            voices = r.json().get("voices", [])
            if not voices: break
            for v in voices:
                vi_voices.append({
                    "voice_id": v["voice_id"],
                    "name": v.get("name", f"vi_{v['voice_id'][:6]}")
                })
            page += 1
        except Exception as e:
            print(f"  ⚠️  Shared voices trang {page}: {e}")
            break

    if not vi_voices:
        print("  ⚠️  Không tìm thấy giọng VI → dùng multilingual fallback")
        vi_voices = [
            {"voice_id": "pNInz6obpgDQGcFmaJgB", "name": "Adam"},
            {"voice_id": "EXAVITQu4vr4xnSDxMaL", "name": "Bella"},
        ]
    return vi_voices


def collect_elevenlabs(texts, out_dir=f"{DATA_DIR}/fake/elevenlabs",
                       n=N_ELEVENLABS):
    valid_keys = [k for k in ELEVENLABS_KEYS if k and not k.startswith("YOUR_")]
    if not valid_keys:
        print("⏭️  ElevenLabs: chưa có key → elevenlabs.io")
        return

    existing = n_existing(out_dir, "*.mp3")
    if existing >= n:
        print(f"✅ ElevenLabs: đã có {existing}/{n} file")
        return

    rotator = KeyRotator(valid_keys, "ElevenLabs")
    print("🔍 Lấy danh sách giọng tiếng Việt từ ElevenLabs...")
    vi_voices   = _get_vi_voices_elevenlabs(rotator.current)
    voice_ids   = [v["voice_id"] for v in vi_voices]
    voice_names = [v["name"]     for v in vi_voices]
    print(f"  {len(vi_voices)} giọng VI: {[v['name'] for v in vi_voices[:6]]}")
    print(f"🔄 ElevenLabs: {n} mẫu | {len(voice_ids)} giọng | {len(valid_keys)} keys")

    for i, text in enumerate(texts[:n]):
        if rotator.is_exhausted():
            print("❌ ElevenLabs: tất cả key hết quota — dừng")
            break

        out_path = f"{out_dir}/elevenlabs_{i:04d}.mp3"
        if os.path.exists(out_path) and os.path.getsize(out_path) > 500:
            continue

        vid   = voice_ids[i % len(voice_ids)]
        vname = voice_names[i % len(voice_names)]

        for attempt in range(len(valid_keys) + 1):
            if rotator.is_exhausted(): break
            try:
                r = requests.post(
                    f"https://api.elevenlabs.io/v1/text-to-speech/{vid}",
                    headers={"xi-api-key": rotator.current,
                             "Content-Type": "application/json"},
                    json={"text": str(text).strip()[:300],
                          "model_id": "eleven_multilingual_v2",
                          "voice_settings": {"stability": 0.5, "similarity_boost": 0.75}},
                    timeout=30
                )
                nr, reason = rotator.should_rotate(r.status_code, r.text)
                if nr:
                    if rotator.rotate(reason):
                        new_vi = _get_vi_voices_elevenlabs(rotator.current)
                        if new_vi:
                            vi_voices   = new_vi
                            voice_ids   = [v["voice_id"] for v in vi_voices]
                            voice_names = [v["name"]     for v in vi_voices]
                            vid   = voice_ids[i % len(voice_ids)]
                            vname = voice_names[i % len(voice_names)]
                    continue

                if r.status_code == 200 and len(r.content) > 500:
                    with open(out_path, "wb") as f: f.write(r.content)
                    print(f"  [{i}] OK | voice={vname} key={rotator.idx}")
                    break
                else:
                    print(f"  ⚠️  [{i}] status={r.status_code}")
                    break
            except Exception as e:
                print(f"  ⚠️  [{i}]: {e}")
                time.sleep(3)

        time.sleep(1.5)

    print(f"✅ ElevenLabs: {n_existing(out_dir, '*.mp3')}/{n} file")
    print(f"   Key còn sống: {rotator.alive_count}/{len(valid_keys)}")

collect_elevenlabs(texts)


## Cell 12 — Nguồn 6: MiniMax TTS (speech-02-turbo)
> Free credits khi đăng ký mới | Multilingual — tự nhận diện tiếng Việt
> Chạy mỗi ngày, tăng `START_IDX_MINIMAX` +50 mỗi ngày

In [ ]:
START_IDX_MINIMAX = 0

def collect_minimax(texts, out_dir=f"{DATA_DIR}/fake/minimax",
                    n=N_MINIMAX, start_idx=0):
    valid_keys = [(k, g) for k, g in MINIMAX_KEYS
                  if k and not k.startswith("YOUR_")]
    if not valid_keys:
        print("⏭️  MiniMax: chưa có key → platform.minimaxi.chat")
        return

    rotator = KeyRotator(valid_keys, "MiniMax")
    batch   = texts[start_idx:start_idx + 50]
    if not batch:
        print("✅ MiniMax: đã xong toàn bộ")
        return

    done_before = n_existing(out_dir, "*.mp3")
    print(f"🔄 MiniMax: câu {start_idx}→{start_idx+len(batch)} | "
          f"{len(valid_keys)} keys | {len(MINIMAX_VOICES)} giọng")

    for i, text in enumerate(batch):
        if rotator.is_exhausted():
            print("❌ MiniMax: tất cả key hết credits — dừng")
            break

        real_idx = start_idx + i
        out_path = f"{out_dir}/minimax_{real_idx:04d}.mp3"
        if os.path.exists(out_path) and os.path.getsize(out_path) > 500:
            continue

        voice   = MINIMAX_VOICES[real_idx % len(MINIMAX_VOICES)]
        success = False

        for attempt in range(len(valid_keys) + 1):
            if rotator.is_exhausted(): break
            api_key, group_id = rotator.current
            try:
                r = requests.post(
                    f"https://api.minimaxi.chat/v1/t2a_v2?GroupId={group_id}",
                    headers={"Authorization": f"Bearer {api_key}",
                             "Content-Type":  "application/json"},
                    json={
                        "model": "speech-02-turbo",
                        "text":  str(text).strip()[:1000],
                        "voice_setting": {
                            "voice_id": voice,
                            "speed":    1.0,
                            "vol":      1.0,
                            "pitch":    0
                        },
                        "audio_setting": {
                            "sample_rate": 16000,
                            "bitrate":     128000,
                            "format":      "mp3"
                        }
                    },
                    timeout=30
                )
                nr, reason = rotator.should_rotate(r.status_code, r.text)
                if nr:
                    if not rotator.rotate(reason): break
                    continue

                if r.status_code == 200:
                    try:
                        data      = r.json()
                        audio_hex = data.get("data", {}).get("audio", "")
                        if audio_hex:
                            with open(out_path, "wb") as f:
                                f.write(bytes.fromhex(audio_hex))
                            print(f"  [{real_idx}] OK | {voice} key={rotator.idx}")
                            success = True; break
                        audio_url = data.get("data", {}).get("audio_file", "")
                        if audio_url:
                            resp = requests.get(audio_url, timeout=20)
                            if resp.status_code == 200 and len(resp.content) > 500:
                                with open(out_path, "wb") as f: f.write(resp.content)
                                print(f"  [{real_idx}] OK (URL) | {voice}")
                                success = True; break
                        print(f"  ⚠️  [{real_idx}] Không có audio: {str(data)[:100]}")
                    except Exception as e:
                        print(f"  ⚠️  [{real_idx}] Parse error: {e}")
                else:
                    print(f"  ⚠️  [{real_idx}] status={r.status_code} {r.text[:80]}")

            except Exception as e:
                print(f"  ⚠️  [{real_idx}]: {e}")
                time.sleep(3)

        time.sleep(1)

    done_after = n_existing(out_dir, "*.mp3")
    print(f"✅ MiniMax hôm nay: +{done_after-done_before} | tổng={done_after}/{n}")
    print(f"   Ngày mai: START_IDX_MINIMAX = {start_idx + 50}")
    print(f"   Key còn sống: {rotator.alive_count}/{len(valid_keys)}")

collect_minimax(texts, start_idx=START_IDX_MINIMAX)


## Cell 13 — Nguồn 7: Zalo AI TTS (1,000 req/ngày miễn phí)
> Đăng ký: zaloai.vn/developers → Tạo App → Lấy API Key
> 5 giọng tiếng Việt: Nam/Nữ Bắc/Nam

In [ ]:
START_IDX_ZALOAI = 0

def collect_zaloai(texts, out_dir=f"{DATA_DIR}/fake/zaloai",
                   n=N_ZALOAI, start_idx=0):
    valid_keys = [k for k in ZALOAI_KEYS if k and not k.startswith("YOUR_")]
    if not valid_keys:
        print("⏭️  Zalo AI: chưa có key → zaloai.vn/developers")
        return

    rotator = KeyRotator(valid_keys, "ZaloAI")
    batch   = texts[start_idx:start_idx + 1000]
    if not batch:
        print("✅ Zalo AI: đã xong toàn bộ")
        return

    done_before = n_existing(out_dir)
    print(f"🔄 Zalo AI: câu {start_idx}→{start_idx+len(batch)} | "
          f"{len(valid_keys)} keys | {len(ZALOAI_SPEAKER_IDS)} giọng")

    session = requests.Session()

    for i, text in enumerate(batch):
        if rotator.is_exhausted():
            print("❌ Zalo AI: tất cả key hết quota — dừng")
            break

        real_idx  = start_idx + i
        out_path  = f"{out_dir}/zaloai_{real_idx:04d}.wav"
        if os.path.exists(out_path) and os.path.getsize(out_path) > 500:
            continue

        speaker_id = ZALOAI_SPEAKER_IDS[real_idx % len(ZALOAI_SPEAKER_IDS)]
        success    = False

        for attempt in range(len(valid_keys) + 1):
            if rotator.is_exhausted(): break
            try:
                r = session.post(
                    "https://api.zalo.ai/v1/tts/synthesize",
                    headers={"apikey":       rotator.current,
                             "Content-Type": "application/x-www-form-urlencoded"},
                    data={
                        "input":       str(text).strip()[:500],
                        "speaker_id":  str(speaker_id),
                        "speed":       "1.0",
                        "encode_type": "1"
                    },
                    timeout=20
                )
                nr, reason = rotator.should_rotate(r.status_code, r.text)
                if nr:
                    if not rotator.rotate(reason): break
                    continue

                if r.status_code == 200:
                    try:
                        data = r.json()
                        audio_url = (data.get("data") or {}).get("url", "")
                        if audio_url:
                            resp = session.get(audio_url, timeout=20)
                            if resp.status_code == 200 and len(resp.content) > 500:
                                with open(out_path, "wb") as f: f.write(resp.content)
                                spk_name = {1:"Nam-Nam",2:"Nữ-Nam",3:"Nam-Bắc",
                                            4:"Nữ-Bắc",5:"Nữ-Bắc2"}.get(speaker_id, str(speaker_id))
                                print(f"  [{real_idx}] OK | {spk_name}")
                                success = True; break
                        ct = r.headers.get("Content-Type", "")
                        if "audio" in ct and len(r.content) > 500:
                            with open(out_path, "wb") as f: f.write(r.content)
                            success = True; break
                        print(f"  ⚠️  [{real_idx}] JSON: {str(data)[:100]}")
                    except Exception as pe:
                        print(f"  ⚠️  [{real_idx}] Parse: {pe}")
                else:
                    print(f"  ⚠️  [{real_idx}] status={r.status_code} {r.text[:60]}")
                    break

            except Exception as e:
                print(f"  ⚠️  [{real_idx}]: {e}")
                time.sleep(3)

        time.sleep(0.5)

    done_after = n_existing(out_dir)
    print(f"✅ Zalo AI hôm nay: +{done_after-done_before} | tổng={done_after}/{n}")
    print(f"   Ngày mai: START_IDX_ZALOAI = {start_idx + len(batch)}")
    print(f"   Key còn sống: {rotator.alive_count}/{len(valid_keys)}")

collect_zaloai(texts, start_idx=START_IDX_ZALOAI)


## Cell 14 — Nguồn 8: gTTS (Google Translate TTS, UNSEEN, hoàn toàn miễn phí)
> **UNSEEN ONLY** — 2 chế độ tốc độ (bình thường + chậm)

In [ ]:
def collect_gtts(texts, out_dir=f"{DATA_DIR}/fake/gtts", n=N_GTTS):
    from gtts import gTTS
    from pydub import AudioSegment

    existing = n_existing(out_dir)
    if existing >= n:
        print(f"✅ gTTS: đã có {existing}/{n} file")
        return

    configs = [
        {"lang": "vi", "slow": False},
        {"lang": "vi", "slow": True},
    ]
    print(f"🔄 gTTS (UNSEEN): {n} mẫu × {len(configs)} chế độ tốc độ")

    for i, text in enumerate(texts[:n]):
        out_path = f"{out_dir}/gtts_{i:04d}.wav"
        if os.path.exists(out_path):
            continue
        cfg = configs[i % len(configs)]
        for attempt in range(3):
            try:
                tts   = gTTS(text=str(text).strip()[:300], **cfg)
                buf   = io.BytesIO()
                tts.write_to_fp(buf)
                buf.seek(0)
                audio = AudioSegment.from_mp3(buf)
                audio = audio.set_frame_rate(TARGET_SR).set_channels(1)
                audio.export(out_path, format="wav")
                break
            except Exception as e:
                if attempt == 2: print(f"  ⚠️  gTTS [{i}]: {e}")
                time.sleep(3 + attempt * 2)
        if i % 100 == 0:
            print(f"  gTTS: {i+1}/{n} | {'chậm' if cfg['slow'] else 'bình thường'}")
        time.sleep(0.5)

    print(f"✅ gTTS (UNSEEN) xong: {n_existing(out_dir)} file → {out_dir}")

collect_gtts(texts)


## Cell 15 — Tiền xử lý: chuẩn hóa toàn bộ audio

In [ ]:
def preprocess_all(n_workers=8):
    print("🔄 Tiền xử lý toàn bộ audio (16kHz mono 5 giây)...")
    tasks = []
    for label in ["real", "fake"]:
        base = Path(f"{DATA_DIR}/{label}")
        if not base.exists(): continue
        for src_dir in base.iterdir():
            if not src_dir.is_dir(): continue
            for f in src_dir.iterdir():
                if f.suffix.lower() not in {".wav", ".mp3", ".m4a"}: continue
                out = (f"{DATA_DIR}/processed/{label}"
                       f"/{src_dir.name}/{f.stem}.wav")
                if not os.path.exists(out):
                    tasks.append((str(f), out))

    if not tasks:
        print("✅ Tất cả file đã được xử lý")
        return

    print(f"  {len(tasks)} file cần xử lý ({n_workers} luồng)...")
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        results = list(ex.map(preprocess_file, tasks))

    ok     = sum(1 for r, _ in results if r)
    errors = [msg for r, msg in results if not r]
    print(f"✅ Xong: {ok}/{len(tasks)} thành công")
    for msg in errors[:5]: print(f"  ⚠️  {msg}")

preprocess_all()


## Cell 16 — Tạo metadata.csv

In [ ]:
def build_metadata():
    print("🔄 Tạo metadata.csv...")
    rows = []

    for label in ["real", "fake"]:
        base = Path(f"{DATA_DIR}/processed/{label}")
        if not base.exists(): continue
        for src_dir in base.iterdir():
            if not src_dir.is_dir(): continue
            source = src_dir.name
            split  = "test_unseen" if source in UNSEEN_SOURCES else "pending"
            for f in sorted(src_dir.glob("*.wav")):
                rows.append({
                    "file_id":   f.stem,
                    "file_path": str(f),
                    "label":     label,
                    "source":    source,
                    "technique": TECHNIQUE_MAP.get(source, "unknown"),
                    "split":     split,
                })

    if not rows:
        print("⚠️  Không có file — chạy Cell 15 trước")
        return

    df = pd.DataFrame(rows)
    seen = df[df["split"] == "pending"].sample(frac=1, random_state=42)
    cut  = int(len(seen) * 0.8)
    df.loc[seen.index[:cut], "split"] = "train"
    df.loc[seen.index[cut:], "split"] = "test_seen"
    df.to_csv(METADATA_FILE, index=False, encoding="utf-8")

    print(f"\n✅ metadata.csv — {len(df)} mẫu tổng cộng")
    print("\n── Theo nguồn + nhãn:")
    print(df.groupby(["source","label"])["file_id"].count()
            .reset_index().rename(columns={"file_id":"count"}).to_string(index=False))
    print("\n── Theo split:")
    print(df["split"].value_counts().to_string())

build_metadata()


## Cell 17 — Kiểm tra chất lượng audio

In [ ]:
def quality_check(sample_size=100):
    if not os.path.exists(METADATA_FILE):
        print("⚠️  Chưa có metadata.csv — chạy Cell 16 trước")
        return

    df  = pd.read_csv(METADATA_FILE)
    smp = df.sample(min(sample_size, len(df)), random_state=42)
    issues = []

    for _, row in smp.iterrows():
        try:
            audio, sr = librosa.load(row["file_path"], sr=None)
            rms  = np.sqrt(np.mean(audio**2))
            peak = np.max(np.abs(audio))
            dur  = len(audio) / sr
            if rms  < 0.005: issues.append(f"Im lặng  : {row['file_id']} RMS={rms:.4f}")
            if peak > 0.99:  issues.append(f"Clipping : {row['file_id']}")
            if dur  < 1.0:   issues.append(f"Quá ngắn : {row['file_id']} {dur:.1f}s")
        except Exception as e:
            issues.append(f"Lỗi đọc  : {row['file_id']}: {e}")

    if issues:
        print(f"⚠️  {len(issues)}/{sample_size} mẫu có vấn đề:")
        for msg in issues: print(f"  - {msg}")
    else:
        print(f"✅ {sample_size} mẫu kiểm tra: tất cả OK")

quality_check()


## Cell 18 — Lưu data (chạy TRƯỚC khi đóng session)
> `/kaggle/working` bị **xóa hoàn toàn** khi session kết thúc!

In [ ]:
import shutil

print("🔄 Đang tạo file backup...")

zip_path = f"{BASE_DIR}/vispoofdb_processed"
shutil.make_archive(zip_path, "zip", f"{DATA_DIR}/processed")

if os.path.exists(METADATA_FILE):
    shutil.copy(METADATA_FILE, f"{BASE_DIR}/metadata_backup.csv")

if os.path.exists(CORPUS_FILE):
    shutil.copy(CORPUS_FILE, f"{BASE_DIR}/corpus_backup.txt")

zip_size = os.path.getsize(f"{zip_path}.zip") / (1024**2)
print(f"\n✅ Backup tại /kaggle/working/:")
print(f"   vispoofdb_processed.zip  ({zip_size:.1f} MB)")
print(f"   metadata_backup.csv")
print(f"   corpus_backup.txt")
print(f"\nDownload 3 file này qua Output panel bên phải Kaggle")
